# 090 — Modelos de difusión

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Forward con alfas concretas

α = [0.9, 0.8, 0.7]; ᾱ₁ = 0.9, ᾱ₂ = 0.72, ᾱ₃ = 0.504.
x₃ = √0.504·2.0 + √0.496·(−1.0) = 1.4199 − 0.7043 ≈ **0.716**.
A t = 3 la señal ya solo aporta la mitad de la varianza (ᾱ₃ ≈ 0.5).


In [ ]:
import math

betas = [0.1, 0.2, 0.3]
x0, eps = 2.0, -1.0
alpha_bar = 1.0
for t, b in enumerate(betas, 1):
    alpha = 1 - b
    alpha_bar *= alpha
    print(f"t={t}: alpha={alpha:.2f}, alpha_bar={alpha_bar:.3f}")
x3 = math.sqrt(alpha_bar) * x0 + math.sqrt(1 - alpha_bar) * eps
print(f"x3 = {x3:.3f}")


## Solución 2 — Señal restante

La varianza se reparte como ᾱ_t señal / (1−ᾱ_t) ruido: **4 % señal, 96 % ruido**.
Estamos cerca del final del forward (t grande): x_t es casi indistinguible de
N(0, I), que es exactamente lo que permite iniciar el reverse desde ruido puro.


In [ ]:
alpha_bar_t = 0.04
print(f"señal: {alpha_bar_t:.0%}, ruido: {1-alpha_bar_t:.0%}")


## Solución 3 — Reconstruir x₀

x̂₀ = (1.114 − √0.28·0.5)/√0.72 = (1.114 − 0.2646)/0.8485 ≈ **1.001** ✔.
Con ᾱ_t = 0.04, el error de ε̂ se amplifica por √(1−ᾱ_t)/√ᾱ_t = √0.96/0.2 ≈ 4.9:
un desvío de 0.1 en ε̂ produce ≈ 0.49 de error en x̂₀. Por eso los primeros pasos del
reverse (t alto) solo esbozan estructura gruesa y los últimos refinan el detalle.


In [ ]:
import math

def x0_hat(x_t, alpha_bar, eps_hat):
    return (x_t - math.sqrt(1 - alpha_bar) * eps_hat) / math.sqrt(alpha_bar)

print(f"x0_hat = {x0_hat(1.114, 0.72, 0.5):.3f}")
amp = math.sqrt(0.96) / math.sqrt(0.04)
print(f"amplificación del error con alpha_bar=0.04: x{amp:.1f}")


## Solución 4 — Contrato del laboratorio

Porque la muestra final depende de toda la trayectoria de muestreo: la semilla fija
los ruidos, pero el scheduler (β_t) y el número de pasos definen la discretización
del proceso; cambiar cualquiera cambia la salida aunque la semilla coincida. El
contrato del laboratorio refleja el mismo principio: semilla + configuración =
evidencia reproducible.


In [ ]:
result = run_lab("generation", seed=90)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)
